# Exploring Tooling with Weights and Biases
Similar to tensorboard, weights and biases is an application that tracks all your training metrics, and performs visualizations for you. This tool allows you to cleanly sort, organize, and visualize your experiments. In this notebook, we will go through an example of how to use wandb.ai and have you practice.

1. Make an account at https://wandb.ai/site

2. pip install wandb

3. wandb login

4. After step 3, please paste your wandb API key


In [1]:
!git clone https://github.com/Berkeley-CS182/cs182fa25_public.git

Cloning into 'cs182fa25_public'...
remote: Enumerating objects: 887, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 887 (delta 28), reused 50 (delta 20), pack-reused 824 (from 1)
Receiving objects: 100% (887/887), 171.00 MiB | 12.86 MiB/s, done.
Resolving deltas: 100% (392/392), done.
Updating files: 100% (376/376), done.


In [2]:
%cd cs182fa25_public/hw06/code

/content/cs182fa25_public/hw06/code


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import wandb
wandb.login(key="901cdada9943c46f375e2a68f350f3dd3d07cbcf")
from architectures import BasicConvNet, ResNet18, MLP

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: menger_wen (menger_wen-university-of-california-berkeley) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Organizing wandb Projects

With each run, you will want to have a set of parameters associated with it. For example, I want to be able to log different hyperparameters that I am using, so let's clearly list them below

In [4]:
project = 'CS182 WANDB.AI Practice Notebook'
learning_rate = 0.01
epochs = 2
architecture ='CNN'
dataset = 'CIFAR-10'
batch_size = 64
momentum = 0.9
log_freq = 20
print_freq = 200
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else "cpu")

### Initializing the Run

In [5]:
wandb.init(
    # set the wandb project where this run will be logged
    project=project,

    # track hyperparameters and run metadata
    config={
    "learning_rate": learning_rate,
    "architecture": architecture,
    "dataset": dataset,
    "epochs": epochs,
    "batch_size": batch_size,
    "momentum": momentum
    }
)

From here on, we have some standard CIFAR training definitions.

In [6]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:01<00:00, 102MB/s]


In [7]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [8]:
net = Net()

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=learning_rate, momentum=momentum)

### Training with wandb

As you can see, similar to tensorboard, each gradient step we will want to log the accuracy and loss. See below for an example.

In [10]:
for epoch in range(epochs):  # loop over the dataset multiple times
    running_loss = 0.0
    running_acc = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        accuracy = torch.mean((torch.argmax(outputs, dim=1) == labels).float()).item() * 100

        # print statistics
        running_acc += accuracy
        running_loss += loss.item()
        if i % log_freq == log_freq - 1:
            wandb.log({'accuracy': accuracy, 'loss': loss.item()})

        if i % print_freq == print_freq - 1:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / print_freq:.5f} accuracy: {running_acc/print_freq:.5f}')
            running_loss = 0.0
            running_acc = 0.0








[1,   200] loss: 2.24518 accuracy: 14.42188
[1,   400] loss: 1.88945 accuracy: 31.21875
[1,   600] loss: 1.68965 accuracy: 38.02344
[2,   200] loss: 1.50270 accuracy: 44.96094
[2,   400] loss: 1.44336 accuracy: 47.67969
[2,   600] loss: 1.40566 accuracy: 49.53906


After we are done with this run, we will want to call
 `wandb.finish()`

In [11]:
wandb.finish()

accuracy,▂▂▁▃▂▃▄▃▃▄▄▅▆▆▅▆▅▆▇▆▅▆▇▆▄█▆▆▆▅▆▅▇█▆▇▆▇▅█
loss,███▇▇▆▄▅▄▄▂▃▂▃▃▄▂▄▃▃▁▂▁▂▂▁▂▄▂▂▁▂▂▁▁▁▁▁▁▂
accuracy,57.8125
loss,1.24757


## Your Task
We will be once again building classifiers for the CIFAR-10. There are various architectures set up for you to use in the `architectures.py` file. Using wandb, please search through 10 different hyperparameter configurations. Examples of choices include: learning rate, batch size, architecture, optimization algorithm, etc. Please submit the hyperparameters that result in the highest accuracies for this classification task. Please then explore wandb for all the visualization that you may need. In addition, feel free to run as many epochs as you like.

In [12]:
def run(params):
    # Initialize a new wandb run with the given project and config
    wandb.init(project=project, config=params)

    # Unpack hyperparameters from params, with defaults if not provided
    learning_rate = params.get('learning_rate', 0.001)
    epochs = params.get('epochs', 5)
    batch_size = params.get('batch_size', 64)
    momentum = params.get('momentum', 0.0)
    architecture = params.get('architecture', 'CNN')  # 'CNN', 'ResNet18', or 'MLP'
    optimizer_name = params.get('optimizer', 'SGD')   # 'SGD' or 'Adam', etc.
    log_freq = params.get('log_freq', 20)
    print_freq = params.get('print_freq', 200)

    # Prepare data transforms (resize for ResNet18 because it expects 224x224 inputs)
    if architecture == 'ResNet18':
        transform = transforms.Compose([
            transforms.Resize(224),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    else:
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    # Load CIFAR-10 dataset with the specified transform and batch size
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Initialize the model based on the chosen architecture
    if architecture == 'CNN':
        net = BasicConvNet()
    elif architecture == 'ResNet18':
        net = ResNet18()
    elif architecture == 'MLP':
        net = MLP()
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    net.to(device)  # move model to GPU if available

    # Set up loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(net.parameters(), lr=learning_rate, momentum=momentum)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(net.parameters(), lr=learning_rate)
    else:
        raise ValueError(f"Unknown optimizer: {optimizer_name}")

    # Training loop
    for epoch in range(epochs):
        net.train()  # set model to training mode
        running_loss = 0.0
        running_acc = 0.0
        for i, (inputs, labels) in enumerate(trainloader):
            # Move inputs and labels to device (GPU/CPU)
            inputs, labels = inputs.to(device), labels.to(device)

            # Zero the parameter gradients
            optimizer.zero_grad()
            # Forward pass
            if architecture == 'MLP':
                # flatten inputs for the MLP
                outputs = net(inputs.view(inputs.size(0), -1))
            else:
                outputs = net(inputs)
            # Compute loss
            loss = criterion(outputs, labels)
            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Compute batch accuracy (%)
            _, predicted = torch.max(outputs.data, 1)
            correct = (predicted == labels).sum().item()
            acc = 100.0 * correct / labels.size(0)

            # Accumulate loss and accuracy
            running_loss += loss.item()
            running_acc += acc

            # Log metrics to wandb every log_freq batches
            if (i + 1) % log_freq == 0:
                wandb.log({
                    'train_loss': loss.item(),
                    'train_accuracy': acc
                })

            # Print statistics every print_freq batches
            if (i + 1) % print_freq == 0:
                avg_loss = running_loss / print_freq
                avg_acc = running_acc / print_freq
                print(f"[Epoch {epoch+1}, Iter {i+1}] loss: {avg_loss:.5f}, accuracy: {avg_acc:.2f}%")
                running_loss = 0.0
                running_acc = 0.0

    # After training, evaluate on the test set
    net.eval()  # set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            if architecture == 'MLP':
                outputs = net(inputs.view(inputs.size(0), -1))
            else:
                outputs = net(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    test_accuracy = 100.0 * correct / total
    print(f"Finished Training. Test Accuracy: {test_accuracy:.2f}%")
    wandb.log({'test_accuracy': test_accuracy})

    # End the wandb run
    wandb.finish()

In [14]:
# Single quick test run
run({
    "architecture": "CNN",        # one of: "CNN", "ResNet18", "MLP"
    "optimizer": "SGD",           # or "Adam"
    "learning_rate": 0.01,
    "momentum": 0.9,              # ignored when optimizer="Adam"
    "batch_size": 64,
    "epochs": 2,
    "log_freq": 20,
    "print_freq": 200
})


[Epoch 1, Iter 200] loss: 2.21656, accuracy: 18.39%
[Epoch 1, Iter 400] loss: 1.86992, accuracy: 31.50%
[Epoch 1, Iter 600] loss: 1.65881, accuracy: 39.80%
[Epoch 2, Iter 200] loss: 1.45825, accuracy: 47.05%
[Epoch 2, Iter 400] loss: 1.42788, accuracy: 48.09%
[Epoch 2, Iter 600] loss: 1.39125, accuracy: 50.02%
Finished Training. Test Accuracy: 51.98%


test_accuracy,▁
train_accuracy,▁▂▂▄▃▃▆▄▄▆▆▆▅▆▇▅▆▇▇▆█▅▇█▇▇▇▆▇▅▇▆▆▇▅▇▇▇██
train_loss,████▇▇▅▆▄▄▄▄▄▄▄▃▃▄▅▂▃▂▁▃▃▄▃▂▃▂▂▂▂▃▃▂▂▄▂▁
test_accuracy,51.98
train_accuracy,50
train_loss,1.32974


This software/tutorial is based on PyTorch, an open-source project available at https://github.com/pytorch/tutorials/

There is a BSD 3-Clause License as seen here: https://github.com/pytorch/tutorials/blob/main/LICENSE